# Практическая работа №2: Обработка выборочных данных. Нахождение точечных оценок параметров распределения

Выполнили студенты гр. 2381 Ваньков Ярослав Сергеевич и Вакуленко Инна Юрьевна. Вариант №20.

## Цель работы

Получение практических навыков нахождения точечных статистических оценок параметров распределения.

## Основные теоретические положения

Условные варианты: $u_i = (x_i - C) / h$, где $C$ — середина одного из интервалов, $h$ — ширина интервала.

Условные эмпирические моменты: $\bar M^*_k = \frac{1}{n}\sum n_i u_i^k$.

Связь с центральными моментами: $\bar m_1 = 0$, $\bar m_2 = h^2(\bar M^*_2 - (\bar M^*_1)^2)$, $\bar m_3 = h^3(\bar M^*_3 - 3\bar M^*_2\bar M^*_1 + 2(\bar M^*_1)^3)$, $\bar m_4 = h^4(\bar M^*_4 - 4\bar M^*_3\bar M^*_1 + 6\bar M^*_2(\bar M^*_1)^2 - 3(\bar M^*_1)^4)$.

Выборочное среднее: $\bar x_в = C + h\bar M^*_1$. Выборочная дисперсия: $D_в = \bar m_2$. СКО: $\sigma_в = \sqrt{D_в}$.

Исправленная дисперсия: $s^2 = \frac{n}{n-1}D_в$. Исправленное СКО: $s = \sqrt{s^2}$.

Коэффициент асимметрии: $\bar A_s = \bar m_3/s^3$. Коэффициент эксцесса: $\bar E = \bar m_4/s^4 - 3$.

Мода для интервального ряда (модальный интервал — с наибольшей частотой): $M_o^* = x_{ниж} + h \frac{n_{мод} - n_{пред}}{2n_{мод} - n_{пред} - n_{след}}$.

Медиана: интервал, где накопленная частота впервые $\ge n/2$: $M_e^* = x_{ниж} + h \frac{n/2 - F_{пред}}{n_{мод}}$.

Коэффициент вариации: $V^* = \frac{\sigma_в}{\bar x_в} \cdot 100\%$.

## Постановка задачи

Для заданных выборочных данных вычислить с использованием метода моментов и условных вариант точечные статистические оценки математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса, моды, медианы и коэффициента вариации исследуемой случайной величины. Полученные результаты содержательно проинтерпретировать.


## Выполнение работы

In [1]:
import pandas as pd
import numpy as np
import math

df = pd.read_csv(r"./sample_120.csv")
x = df["sepal_length"].to_numpy()
n = len(x)
df.head()

,sepal_length,sepal_width
0,5.2,3.4
1,4.6,3.4
2,5.9,3.2
3,6.0,2.2
4,4.9,2.4


In [2]:
k = math.ceil(1 + 3.322 * math.log10(n))
xmin, xmax = float(x.min()), float(x.max())
h = (xmax - xmin) / k
bins = np.array([xmin + i*h for i in range(k)] + [xmax])
bins[-1] = xmax + 1e-12
cats = pd.cut(x, bins=bins, right=False, include_lowest=True)
m = cats.value_counts().sort_index().to_numpy()
mid = (bins[:-1] + bins[1:]) / 2
int_df = pd.DataFrame({
    "i": np.arange(1, k+1),
    "[x_i; x_{i+1})": [f"[{bins[i]:.4f}; {bins[i+1]:.4f})" for i in range(k)],
    "x_tilde_i": np.round(mid, 6),
    "m_i": m,
})
int_df["m_tilde_i"] = int_df["m_i"] / n
int_df["m_i_cum"] = int_df["m_i"].cumsum()
int_df["m_tilde_i_cum"] = int_df["m_tilde_i"].cumsum()
int_df

,i,[x_i; x_{i+1}),x_tilde_i,m_i,m_tilde_i,m_i_cum,m_tilde_i_cum
0,1,[4.3000; 4.7250),4.5125,9,0.075000,9,0.075000
1,2,[4.7250; 5.1500),4.9375,26,0.216667,35,0.291667
2,3,[5.1500; 5.5750),5.3625,16,0.133333,51,0.425000
3,4,[5.5750; 6.0000),5.7875,20,0.166667,71,0.591667
4,5,[6.0000; 6.4250),6.2125,20,0.166667,91,0.758333
5,6,[6.4250; 6.8500),6.6375,16,0.133333,107,0.891667
6,7,[6.8500; 7.2750),7.0625,7,0.058333,114,0.950000
7,8,[7.2750; 7.7000),7.4875,6,0.050000,120,1.000000


**Задание.** Для середин интервального ряда, полученного в практической работе №1, вычислить условные варианты. Заполнить табл. 1 (в последней строке Sum — суммы столбцов; ячейки, отмеченные прочерком, заполнять не надо). Провести контроль вычислений.

**Таблица 1. Условные варианты**

In [3]:
C = mid[k // 2 - 1]
u = np.round((mid - C) / h).astype(int)
n_i = int_df["m_i"].values
t1 = pd.DataFrame({
    "i": np.arange(1, k+1),
    "x_i": mid,
    "n_i": n_i,
    "u_i": u,
    "n_i*u_i": n_i * u,
    "n_i*u_i^2": n_i * (u**2),
    "n_i*u_i^3": n_i * (u**3),
    "n_i*u_i^4": n_i * (u**4),
    "n_i*(u_i+1)^4": n_i * ((u+1)**4),
})
row_sum = pd.DataFrame([{
    "i": "Sum",
    "x_i": "",
    "n_i": t1["n_i"].sum(),
    "u_i": "-",
    "n_i*u_i": t1["n_i*u_i"].sum(),
    "n_i*u_i^2": t1["n_i*u_i^2"].sum(),
    "n_i*u_i^3": t1["n_i*u_i^3"].sum(),
    "n_i*u_i^4": t1["n_i*u_i^4"].sum(),
    "n_i*(u_i+1)^4": t1["n_i*(u_i+1)^4"].sum(),
}])
tab1 = pd.concat([t1, row_sum], ignore_index=True)
tab1

,i,x_i,n_i,u_i,n_i*u_i,n_i*u_i^2,n_i*u_i^3,n_i*u_i^4,n_i*(u_i+1)^4
0,1,4.5125,9,-3,-27,81,-243,729,144
1,2,4.9375,26,-2,-52,104,-208,416,26
2,3,5.3625,16,-1,-16,16,-16,16,0
3,4,5.7875,20,0,0,0,0,0,20
4,5,6.2125,20,1,20,20,20,20,320
5,6,6.6375,16,2,32,64,128,256,1296
6,7,7.0625,7,3,21,63,189,567,1792
7,8,7.4875,6,4,24,96,384,1536,3750
8,Sum,,120,-,2,444,254,3540,7348


По таблице: суммы по столбцам сходятся, контроль по (u+1)^4 выполняется — значит, условные варианты и произведения посчитаны верно. Эти суммы дальше пойдут в расчёт моментов.

In [4]:
S_nu = tab1.loc[tab1["i"] == "Sum"].iloc[0]
check = (S_nu["n_i*(u_i+1)^4"] - (S_nu["n_i*u_i^4"] + 4*S_nu["n_i*u_i^3"] + 6*S_nu["n_i*u_i^2"] + 4*S_nu["n_i*u_i"] + S_nu["n_i"])) < 1e-6
print("Контроль (u+1)^4:", "OK" if check else "Ошибка")

Контроль (u+1)^4: OK


**Задание.** Вычислить условные эмпирические моменты $\bar M^*_k$ через условные варианты. С помощью них вычислить центральные эмпирические моменты $\bar m_k$. Результаты занести в табл. 2.

**Таблица 2. Условные и центральные моменты**

По табл. 2: m_1 получается ноль — так и должно быть для центральных моментов. Остальные m_k дальше используем для среднего, дисперсии и для асимметрии с эксцессом.

In [5]:
M1 = t1["n_i*u_i"].sum() / n
M2 = t1["n_i*u_i^2"].sum() / n
M3 = t1["n_i*u_i^3"].sum() / n
M4 = t1["n_i*u_i^4"].sum() / n
m1 = 0
m2 = (M2 - M1**2) * h**2
m3 = (M3 - 3*M2*M1 + 2*M1**3) * h**3
m4 = (M4 - 4*M3*M1 + 6*M2*M1**2 - 3*M1**4) * h**4
tab2 = pd.DataFrame({"k": [1,2,3,4], "M*_k": [M1,M2,M3,M4], "m_k": [m1,m2,m3,m4]})
tab2

,k,M*_k,m_k
0,1,0.016667,0.000000
1,2,3.700000,0.668262
2,3,2.116667,0.148286
3,4,29.500000,0.958046


### Выборочное среднее и дисперсия

In [6]:
x_bar_std = (t1["x_i"] * t1["n_i"]).sum() / n
x_bar_cond = C + h * M1
D_v_std = (t1["n_i"] * (t1["x_i"] - x_bar_std)**2).sum() / n
D_v_cond = m2
sigma_v = np.sqrt(D_v_std)
print("x_bar (стандарт):", round(x_bar_std, 6), "| (условные):", round(x_bar_cond, 6))
print("D_v (стандарт):", round(D_v_std, 6), "| (условные):", round(D_v_cond, 6))
print("sigma_v:", round(sigma_v, 6))

x_bar (стандарт): 5.794583 | (условные): 5.794583
D_v (стандарт): 0.668262 | (условные): 0.668262
sigma_v: 0.817473


### Исправленная дисперсия и СКО

In [7]:
s2 = n / (n - 1) * D_v_std
s_val = np.sqrt(s2)
print("s^2:", round(s2, 6), "| s:", round(s_val, 6))
print("D_v (смещённая):", round(D_v_std, 6), "; s^2 (исправленная) > D_v при n конечном.")

s^2: 0.673878 | s: 0.820901
D_v (смещённая): 0.668262 ; s^2 (исправленная) > D_v при n конечном.


### Асимметрия и эксцесс

In [8]:
A_s = m3 / (s_val**3)
E_val = m4 / (s_val**4) - 3
print("A_s:", round(A_s, 4), "| E:", round(E_val, 4))

A_s: 0.2681 | E: -0.8903


### Мода, медиана, коэффициент вариации

In [9]:
idx_mod = int_df["m_i"].idxmax()
n_mod = int_df.loc[idx_mod, "m_i"]
n_prev = int_df.loc[idx_mod-1, "m_i"] if idx_mod > 0 else 0
n_next = int_df.loc[idx_mod+1, "m_i"] if idx_mod < k-1 else 0
x_low = bins[idx_mod]
M_o = x_low + h * (n_mod - n_prev) / (2*n_mod - n_prev - n_next) if (2*n_mod - n_prev - n_next) != 0 else mid[idx_mod]
cum = int_df["m_i_cum"].values
idx_med = np.searchsorted(cum, n/2)
F_prev = cum[idx_med-1] if idx_med > 0 else 0
M_e = bins[idx_med] + h * (n/2 - F_prev) / int_df.loc[idx_med, "m_i"]
V_star = sigma_v / x_bar_std * 100
print("M_o*:", round(M_o, 4), "| M_e*:", round(M_e, 4), "| V* (%):", round(V_star, 2))

M_o*: 4.9926 | M_e*: 5.7662 | V* (%): 14.11


## Выводы

Работали с интервальным рядом по sepal_length (120 наблюдений). Посчитали условные варианты, заполнили обе таблицы, контроль сошёлся. Среднее и дисперсия, полученные обычным способом и через условные моменты, совпали; s² и s чуть больше смещённых оценок, как и ожидается. По коэффициентам асимметрии и эксцесса видно, как распределение отличается от симметричного и от нормального. Мода, медиана и V* дают доп. ориентиры по центру и разбросу данных.